In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
PROCESSED_DATA_PATH = Path("data/processed")

df_repos = pd.read_parquet(PROCESSED_DATA_PATH / "repositories.parquet")
df_wfs = pd.read_parquet(PROCESSED_DATA_PATH / "workflows.parquet")
df_bodies = pd.read_parquet(PROCESSED_DATA_PATH / "workflow_bodies.parquet")

In [ ]:
# Conteo de workflows por repositorio
wf_counts = df_wfs.groupby("repository_id").size().rename("wf_count")

summary_stats = pd.DataFrame(
    {
        "Métrica": ["Mínimo", "Máximo", "Media", "Mediana"],
        "Valor": [
            wf_counts.min(),
            wf_counts.max(),
            wf_counts.mean(),
            wf_counts.median(),
        ],
    }
)

plt.figure(figsize=(8, 4))
sns.histplot(wf_counts, bins=20, kde=True, color="skyblue")
plt.title("Distribución de Archivos Markdown por Repositorio")
plt.xlabel("Cantidad de Archivos Markdown")
plt.ylabel("Frecuencia (Repositorios)")
plt.tight_layout()
plt.show()

display(summary_stats)

In [ ]:
# 1. Porcentaje de presencia de campos conocidos
total_wfs = len(df_wfs)
fields_presence = pd.DataFrame(
    {
        "Campo": ["title", "description", "engine"],
        "Presentes": [
            df_wfs["title"].notnull().sum(),
            df_wfs["description"].notnull().sum(),
            df_wfs["engine"].notnull().sum(),
        ],
    }
)
fields_presence["Porcentaje (%)"] = (
    fields_presence["Presentes"] / total_wfs
) * 100

# 2. Análisis del atributo Motor de IA (engine)
engine_counts = (
    df_wfs["engine"].fillna("No declarado").value_counts().reset_index()
)
engine_counts.columns = ["Motor de IA", "Frecuencia"]

plt.figure(figsize=(10, 4))
sns.barplot(data=engine_counts.head(10), x="Frecuencia", y="Motor de IA", palette="mako")
plt.title("Top 10 Motores de IA Declarados en Frontmatter")
plt.tight_layout()
plt.show()

display(fields_presence)
display(engine_counts.head(10))

In [ ]:
# Calcular longitud en palabras del body
df_bodies["word_count"] = df_bodies["body_markdown"].apply(
    lambda x: len(str(x).split()) if pd.notnull(x) else 0
)

body_stats = pd.DataFrame(
    {
        "Métrica": ["Mínimo", "Máximo", "Media", "Mediana"],
        "Valor": [
            df_bodies["word_count"].min(),
            df_bodies["word_count"].max(),
            df_bodies["word_count"].mean(),
            df_bodies["word_count"].median(),
        ],
    }
)

# Identificación de vacíos y extremos
empty_bodies = df_bodies[df_bodies["word_count"] == 0]
print(f"Archivos con body vacío: {len(empty_bodies):,}")

plt.figure(figsize=(8, 4))
sns.boxplot(x=df_bodies["word_count"], color="lightgreen")
plt.title("Distribución de Longitud de Palabras en el Body")
plt.xlabel("Cantidad de Palabras")
plt.tight_layout()
plt.show()

display(body_stats)

In [ ]:
# Pregunta 1: Engine vs Longitud del Body (Cruce de workflows + workflow_bodies)
merged_wf_body = df_wfs.merge(
    df_bodies, left_on="id", right_on="workflow_id"
)
engine_len = (
    merged_wf_body.groupby("engine")["word_count"]
    .agg(["count", "mean", "median"])
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=merged_wf_body[
        merged_wf_body["engine"].isin(engine_counts["Motor de IA"].head(5))
    ],
    x="engine",
    y="word_count",
    palette="Set2",
)
plt.yscale("log")  # Escala logarítmica para lidiar con outliers
plt.title("Relación entre Motor de IA y Longitud del Body (Escala Log)")
plt.xlabel("Motor de IA")
plt.ylabel("Palabras en Body (Log)")
plt.tight_layout()
plt.show()